# Day 5 – UAT Signoff – Healthcare

Simple UAT checks for the healthcare project.

In [0]:
from pyspark.sql import functions as F

CATALOG = "capstone_8hospital"
SILVER = f"{CATALOG}.silver_schema"
GOLD = f"{CATALOG}.gold_schema"

fail = 0

print("UAT started")

## 1. Patient count check

In [0]:
silver_patients = spark.table(f"{SILVER}.silver_patients").count()
gold_patients = spark.table(f"{GOLD}.dim_patient").count()

print("Silver patients:", silver_patients)
print("Gold patients:", gold_patients)

if silver_patients == gold_patients:
    print("PASS - patient count")
else:
    print("FAIL - patient count")
    fail += 1

## 2. Encounter count check

In [0]:
silver_encounters = spark.table(f"{SILVER}.silver_patient_encounters").count()
gold_encounters = spark.table(f"{GOLD}.fact_patient_encounter").count()

print("Silver encounters:", silver_encounters)
print("Gold encounters:", gold_encounters)

if silver_encounters == gold_encounters:
    print("PASS - encounter count")
else:
    print("FAIL - encounter count")
    fail += 1

## 3. Age-band check

In [0]:
patients = spark.table(f"{GOLD}.dim_patient")

age_check = (patients
    .groupBy("age_band")
    .agg(F.countDistinct("patient_key").alias("patient_count"))
    .orderBy("age_band"))

display(age_check)

bad_age = patients.filter(F.col("age_band").isNull()).count()
print("Missing age_band:", bad_age)

if bad_age == 0:
    print("PASS - age_band")
else:
    print("FAIL - age_band")
    fail += 1

## 4. Data quality / null check

In [0]:
bad_patient_key = patients.filter(F.col("patient_key").isNull()).count()
bad_encounter_key = spark.table(f"{GOLD}.fact_patient_encounter").filter(
    F.col("encounter_key").isNull()
).count()

print("Missing patient keys:", bad_patient_key)
print("Missing encounter keys:", bad_encounter_key)

if bad_patient_key == 0 and bad_encounter_key == 0:
    print("PASS - data quality")
else:
    print("FAIL - data quality")
    fail += 1

## 5. Final UAT signoff

In [0]:
if fail == 0:
    print("UAT status: SUCCESS")
else:
    print("UAT status: FAILED")

print("Failed checks:", fail)

assert fail == 0, "UAT signoff failed"